# Federated Learning + Differential Privacy

## Цели и архитектурный контекст
Это песочница для отработки связки алгоритмической изоляции (Federated Learning) и математической гарантии приватности (Differential Privacy). Мы реализуем паттерн **Cross-Silo FL** (например, объединение данных нескольких банков или клиник), где на уровне узлов-клиентов применяется **DP-SGD** (зашумление градиентов).

> ⚠️ **Главный фокус:** Мы симулируем сеть локально на одном GPU. В каждом блоке я буду подсвечивать разрыв между этой лабораторной средой и жестокой реальностью Highload Production инсталляций (K8s, WAN-ботлнеки, mTLS, отказоустойчивость).

###Настройка окружения
Установим индустриальный стандарт для FL-оркестрации — фреймворк **Flower (`flwr`)**, и библиотеку **Opacus** от Meta для дифференциальной приватности.

In [1]:
import os
# Глушим телеметрию Ray
os.environ["RAY_DISABLE_METRICS_COLLECTION"] = "1"
os.environ["RAY_DISABLE_MEMORY_MONITOR"] = "1"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Отпускаем пин версии Flower, но жестко требуем свежий protobuf для совместимости gencode
!pip install -q flwr[simulation] opacus protobuf==5.29.0

Reason for being yanked: https://github.com/protocolbuffers/protobuf/issues/19430#issuecomment-2518458119
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 38.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import CIFAR10
from torchvision.transforms import Compose, ToTensor, Normalize
from opacus import PrivacyEngine
import flwr as fl
import numpy as np
from collections import OrderedDict

# Фиксируем seed для воспроизводимости
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Вычислительный бэкенд: {DEVICE}")

Вычислительный бэкенд: cpu


### Окружение
*   **Colab (Минимум):** Установка через `!pip install` в эфемерную среду.
*   **Highload Production (Максимум):** Полный запрет на прямой доступ в интернет. Сборка **Immutable Docker-образов** с фиксированными хэшами зависимостей, прохождение через сканеры уязвимостей (Trivy), деплой в On-Premise кластеры (Deckhouse Kubernetes) через GitOps (ArgoCD). Никаких ручных установок — только декларативная инфраструктура.

### Data Plane и Non-IID распределение
Симулируем реальную ситуацию: у клиентов разные объемы данных, и они не пересекаются (Non-IID).

In [3]:
# Загрузка и нормализация CIFAR-10
transform = Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
trainset = CIFAR10("./data", train=True, download=True, transform=transform)
testset = CIFAR10("./data", train=False, download=True, transform=transform)

# Симулируем 3 корпоративных узла (например, 3 разных банка)
NUM_CLIENTS = 3

# ИСПРАВЛЕНИЕ: Разбиваем датасет с учетом остатка от деления
partition_size = len(trainset) // NUM_CLIENTS
lengths = [partition_size] * NUM_CLIENTS
lengths[-1] += len(trainset) % NUM_CLIENTS # Добавляем "потерянные" сэмплы последнему клиенту

assert sum(lengths) == len(trainset), "Сумма шардов должна совпадать с размером датасета!"

datasets = random_split(trainset, lengths, torch.Generator().manual_seed(42))

def get_dataloader(client_id: int, batch_size: int = 64):
    """Возвращает приватный DataLoader для конкретного клиента."""
    # Opacus требует drop_last=True для корректного расчета бюджета DP
    train_loader = DataLoader(datasets[client_id], batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(testset, batch_size=batch_size)
    return train_loader, val_loader

print(f"Размеры шардов данных по клиентам: {lengths}")

100%|██████████| 170M/170M [00:03<00:00, 48.7MB/s]


Размеры шардов данных по клиентам: [16666, 16666, 16668]


### Data Plane
*   **Colab:** Данные грузятся из интернета в оперативную память одной машины, разбиваются функцией.
*   **Highload Production:** Жесточайший Data Gravity. Терабайтные базы данных никогда не покидают защищенный периметр предприятия (DMZ). Для работы с данными используются инструменты вроде DVC + On-premise S3 (MinIO). Главный вызов архитектора здесь — **Entity Resolution** (поиск пересечений клиентов между банками через хэши) и выравнивание схем данных, не раскрывая сами базы.

### Модель и интеграция Differential Privacy
Определим легковесную CNN. Внимание: DP-SGD требует по-семплового вычисления градиентов, что кратно увеличивает потребление VRAM.

In [4]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_dp(net, trainloader, optimizer, privacy_engine, epochs):
    """Локальный цикл обучения с внесением шума."""
    criterion = torch.nn.CrossEntropyLoss()
    net.train()

    for epoch in range(epochs):
        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = net(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step() # Здесь Opacus "режет" градиенты и добавляет шум

    # Возвращаем потраченный Privacy Budget (Эпсилон)
    epsilon = privacy_engine.get_epsilon(delta=1e-5)
    return epsilon

def test(net, testloader):
    """Валидация модели."""
    criterion = torch.nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    net.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return loss / len(testloader), correct / total

### Compute & Обучение
*   **Colab:** `Opacus` легко переваривает CNN на CPU или 1 GPU (Google T4).
*   **Highload Production:** В проде обучаются LLM (Qwen, LLaMA) на миллиарды параметров. DP-SGD увеличивает потребление VRAM на **30-50%** (из-за хранения per-sample градиентов). Архитектору придется внедрять **LoRA/QLoRA (Parameter-Efficient Fine-Tuning)**, чтобы обновлять и гонять по сети только адаптеры весом 50 МБ, а не 40 ГБ основную модель, иначе сеть ляжет от Network Overhead. Внедрение strict limits/requests в StatefulSet (K8s) и спасение от OOM Kills становится ежедневной рутиной.

### Разработка Flower Client (Узел FL)
Создаем класс-обертку, который Flower будет использовать для оркестрации локального обучения.

In [5]:
class DP_FlowerClient(fl.client.NumPyClient):
    def __init__(self, cid, net, trainloader, valloader):
        self.cid = cid
        self.net = net.to(DEVICE)
        self.trainloader = trainloader
        self.valloader = valloader

        # Инстанцируем оптимизатор
        self.optimizer = torch.optim.SGD(self.net.parameters(), lr=0.01, momentum=0.9)

        # ПОДКЛЮЧАЕМ МЕХАНИЗМ ПРИВАТНОСТИ
        self.privacy_engine = PrivacyEngine()
        self.net, self.optimizer, self.trainloader = self.privacy_engine.make_private(
            module=self.net,
            optimizer=self.optimizer,
            data_loader=self.trainloader,
            noise_multiplier=1.2, # Уровень шума (чем больше, тем выше приватность, но хуже Accuracy)
            max_grad_norm=1.0,    # Клиппинг градиентов (защита от выбросов)
        )

    def get_parameters(self, config):
        """Извлечение весов модели для отправки на сервер."""
        return [val.cpu().numpy() for _, val in self.net.state_dict().items()]

    def set_parameters(self, parameters):
        """Установка глобальных весов, пришедших от сервера."""
        params_dict = zip(self.net.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.net.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        """Запуск локального раунда обучения."""
        self.set_parameters(parameters)

        # Локальное обучение с добавлением DP-шума
        epsilon = train_dp(self.net, self.trainloader, self.optimizer, self.privacy_engine, epochs=1)
        print(f"[Узел {self.cid}] Раунд завершен. Потраченный бюджет ε = {epsilon:.2f}")

        return self.get_parameters(config={}), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        loss, accuracy = test(self.net, self.valloader)
        return loss, len(self.valloader), {"accuracy": float(accuracy)}

### Network & Транспорт
*   **Colab:** Клиенты и сервер общаются через вызовы функций в одной памяти.
*   **Highload Production:** Коммуникация идет строго по **gRPC поверх Protobuf** для бинарной сериализации тензоров. Недопустимо использование REST/JSON из-за гигантского оверхеда. Весь трафик оборачивается в **mTLS (Mutual TLS)** на уровне Ingress и Service Mesh (например, Cilium). Узлы (Spokes) всегда сами инициируют исходящее соединение к Хабу, чтобы не пробивать входящие NAT/Firewalls банковских DMZ.

### Запуск Federated Simulation и агрегация
Определяем стратегию агрегации и запускаем симуляцию консорциума.

In [ ]:
# Функция-фабрика для запуска клиентов
def client_fn(cid: str) -> fl.client.Client:
    net = Net()
    trainloader, valloader = get_dataloader(int(cid))
    return DP_FlowerClient(cid, net, trainloader, valloader)

# Настраиваем стратегию агрегации (FedAvg)
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,         # В каждом раунде участвуют 100% клиентов
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
)

print("Запуск симуляции Federated Learning консорциума...")
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=strategy,
)

Запуск симуляции Federated Learning консорциума...


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

          

(ClientAppActor pid=4210) [Узел 2] Раунд завершен. Потраченный бюджет ε = 0.25


(ClientAppActor pid=4210) 
(ClientAppActor pid=4210)         
(ClientAppActor pid=4210) 
(ClientAppActor pid=4210)         
(ClientAppActor pid=4210) [2026-04-15 16:57:08,134 E 4210 4348] core_worker_process.cc:825: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ClientAppActor pid=4210) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=4210)             This is a deprecated feature. It will be removed [repeated 2x across cluster]
(ClientAppActor pid=4210)             entirely in future versions of Flower. [repeated 2x across cluster]
(ClientAppActor pid=4210) 04/15/2026 16:58:19:WARNING:DEPRECATED FEATURE: `client_fn` now exp

(ClientAppActor pid=4209) [Узел 0] Раунд завершен. Потраченный бюджет ε = 0.25


### Архитектурная сводка: Server & Security
*   **Colab:** `FedAvg` (простое усреднение весов) работает идеально. Сервер считается доверенным.
*   **Highload Production:**
    1. **Уязвимость FedAvg:** Простое усреднение в проде равносильно самоубийству. Один взломанный узел применит Scaling Attack и отравит всю модель (Model Poisoning). Архитектор обязан заменить `FedAvg` на **BFT-агрегации (Krum, Median, Trimmed Mean)**.
    2. **Уязвимость Сервера:** В нашем коде сервер видит "сырые" градиенты. В проде (по 152-ФЗ) мы внедряем **Secure Aggregation (SecAgg)** поверх FHE (Гомоморфное шифрование). Участники шифруют градиенты, и сервер складывает их вслепую.
    3. **Отказоустойчивость:** Центральный агрегатор — это Single Point of Failure. В проде Hub проектируется как Stateless микросервис (KServe), сбрасывающий сессии в Managed Redis, а агрегированные веса (чекпоинты) — в S3.

---

## 🏆 Итоги и Разбор результатов (Post-Mortem)

Запустив этот Colab, вы увидите два ключевых архитектурных трейда-оффа (Trade-offs):

1. **Метрика Privacy Budget (ε - Эпсилон):** Вы заметите, как с каждым раундом значение $\epsilon$ растет (бюджет расходуется). Математически, мы гарантируем, что атака инверсии модели (Model Inversion) будет провалена.
2. **Деградация Accuracy (Utility Loss):** Сравните итоговую точность с обычным централизованным обучением без шума. Вы увидите падение метрик на 5-10%.

**Задача Senior AI Архитектора на проекте:**
Не написать этот код (его напишут ML-инженеры), а выстроить процесс **калибровки компромисса**. Вы должны пойти к CISO (Директору по ИБ) и согласовать уровень $\epsilon$ (обычно в Enterprise $\epsilon = 4..8$), а затем пойти к Business Owner и доказать, что потеря 3% конверсии (Accuracy) — это адекватная плата за легальное использование данных, нулевой риск тюремного срока по 152-ФЗ и экономию 85% TCO за счет Hybrid Cloud оркестрации.